In [1]:
from pathlib import Path

import numpy as np
import scipy.optimize as spo

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.utils import analysis, visualization

In [2]:
settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_logfile.log"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.005,
        tau=100,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

In [3]:
posterior_builder = builder.PosteriorBuilder(settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
visualization.visualize_data_points(
    mesh=additional_output.pv_mesh,
    observation_inds=additional_output.observation_inds,
)

Widget(value='<iframe src="http://localhost:45961/index.html?ui=P_0x7f7137a7f230_0&reconnect=auto" class="pyvi…

In [ ]:
optimizer_options = {
    "disp": True,
    "maxiter": 2000,
    "ftol": 1e-6,
    "gtol": 1e-6,
    "maxls": 100,
}
map_estimate = spo.minimize(
    fun=posterior.evaluate_cost,
    jac=posterior.evaluate_gradient,
    x0=np.zeros(additional_output.pv_mesh.number_of_points),
    method="L-BFGS-B",
    options=optimizer_options,
)
print(map_estimate)
np.save("../results/map_estimate.npy", map_estimate.x)

In [ ]:
map_parameter = np.load("../results/map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_parameter,
    posterior=posterior,
    additional_output=additional_output,
)

In [ ]:
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.prior_mean_parameter,
    circular=False,
)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.ground_truth_parameter,
    circular=False,
)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.map_parameter,
    circular=False,
)

In [ ]:
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_lat_truth_prior,
    circular=False,
)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_lat_truth_map,
    circular=False,
)